# Introduction, Sample Surveys and Questionnaire Design

**STA 503 1.0 Sampling Techniques** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta503.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [The inferential purpose of a survey](#theory-01)
- [Census and sample](#theory-02)
- [Observational and experimental studies](#theory-03)
- [Planning steps](#theory-04)
- [Total survey error](#theory-05)
- [Questionnaire structure](#theory-06)
- [Question types and measurement](#theory-07)
- [Pilot, fieldwork and processing](#theory-08)
- [Reporting](#theory-09)

## Detailed theory · Survey foundations and questionnaire design

<a id="theory-01"></a>

### The inferential purpose of a survey
A survey collects information to answer stated questions about a target population. Distinguish the target population, observational element, sampling unit, reporting unit and frame. A household can be sampled while a person reports on its members. An estimate applies to the defined population and period, not every superficially similar group.

<a id="theory-02"></a>

### Census and sample
A census attempts every unit; a sample observes selected units. Sampling can reduce cost, delay and burden and sometimes improve measurement control. A census removes selection variation for the fully observed finite population, but not undercoverage, measurement, processing or nonresponse error. Destructive testing may make a census impossible.

<a id="theory-03"></a>

### Observational and experimental studies
Observational surveys measure without assigning the exposure of interest. Experiments impose treatments. Random selection concerns representation; treatment assignment concerns comparability. Cause-and-effect conclusions need more justification than a representative observational sample.

<a id="theory-04"></a>

### Planning steps
Specify objectives, target population and reporting period; identify the variables and required precision; inspect existing data; construct a frame; choose a design; develop and pilot the instrument; plan fieldwork and nonresponse handling; collect, process, analyse and report. The analysis plan should inform question design and sample size before data collection.

<a id="theory-05"></a>

### Total survey error
Sampling error is variation from observing a sample rather than the whole frame population under a design. Nonsampling errors include frame omissions, duplicate units, missing responses, interviewer effects, recall, ambiguous questions, incorrect coding and data entry. Increasing n mainly reduces sampling variation; it does not automatically reduce systematic bias.

<a id="theory-06"></a>

### Questionnaire structure
Use clear objectives and one concept per question. Specify units and reference periods. Ask neutral questions in language respondents understand. Avoid leading, double-barrelled, loaded or assumption-heavy questions. Place simple relevant questions first, use logical sections and routing, and explain confidentiality and purpose. Response options should be mutually exclusive and collectively adequate.

<a id="theory-07"></a>

### Question types and measurement
Closed questions standardize coding but can force unsuitable categories; open questions permit nuance but require a coding plan. Ordinal response scales should have a clear direction and balanced labels. A self-completed questionnaire and an interviewer-administered schedule have different practical strengths and sources of bias.

<a id="theory-08"></a>

### Pilot, fieldwork and processing
Test comprehension, timing, skip logic and coding on a pilot. Train interviewers, standardize contacts, track dispositions and preserve the difference between refusal, not contacted and ineligible. Validate ranges and cross-field consistency without silently inventing missing responses. Keep a codebook and correction log.

<a id="theory-09"></a>

### Reporting
Document frame coverage, recruitment, design, achieved sample, response handling, weighting and uncertainty. A response rate alone does not quantify nonresponse bias. Link each reported table or estimate to an objective and disclose limitations.

## Concepts covered

**Course points:** Population and frame; questionnaire design; survey stages; experimental vs observational studies; coverage and nonresponse.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Survey planning: trace the objective to the questionnaire

### Question and assumptions

Plan an estimate of mean weekly household water use in a stated district and month.

### Procedure

1. Define the target population and unit.
2. Identify a frame, sample procedure, measurements and nonresponse follow-up.
3. Show which design choice protects each conclusion.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Step | Concrete choice | Reason / check |
| --- | --- | --- |
| Objective | Mean weekly household water use | Specify period and units |
| Population | All occupied households in the district | Include unregistered households if in scope |
| Unit | Household | Do not confuse with an individual respondent |
| Frame | Updated household address list | Check omissions and duplicates |
| Selection | Random addresses from the frame | Record inclusion probabilities |
| Question | Meter use over a stated seven-day interval | Use consistent units and dates |
| Nonresponse | Record outcome and make planned follow-up | Do not replace by convenient neighbours |
| Estimate | Design-appropriate mean and SE | Match weights and variance to sampling design |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


show_table(["Step","Concrete choice","Reason / check"],[
["Objective","Mean weekly household water use","Specify period and units"],
["Population","All occupied households in the district","Include unregistered households if in scope"],
["Unit","Household","Do not confuse with an individual respondent"],
["Frame","Updated household address list","Check omissions and duplicates"],
["Selection","Random addresses from the frame","Record inclusion probabilities"],
["Question","Meter use over a stated seven-day interval","Use consistent units and dates"],
["Nonresponse","Record outcome and make planned follow-up","Do not replace by convenient neighbours"],
["Estimate","Design-appropriate mean and SE","Match weights and variance to sampling design"]])


### Decision and interpretation

A sampling frame is a practical list and can differ from the target population. A precise calculation cannot remove frame or response bias.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
N=20000; remote=rng.binomial(1,.25,N)
time=np.where(remote==1,rng.gamma(4,12,N),rng.gamma(3,6,N))
p_answer=1/(1+np.exp((time-60)/18)); answered=rng.random(N)<p_answer
print('population mean',time.mean().round(2),'respondent mean',time[answered].mean().round(2),'response rate',answered.mean().round(2))
plt.hist(time,bins=40,density=True,alpha=.5,label='all'); plt.hist(time[answered],bins=40,density=True,alpha=.5,label='respondents'); plt.legend(); plt.show()

### Practice set · Survey design
1. Write an estimand for average weekly clinic travel time among adults in a named district.
2. Improve this question: “You agree the new, excellent clinic is easier to reach, right?”
3. Identify likely coverage error and nonresponse error in an online-only survey of all residents.
4. Explain how random assignment differs from random sampling.

<details><summary>Answer sketches</summary>
1. Specify mean travel minutes, population, place, and period. 2. “How many minutes did your most recent trip to a clinic take?” with a defined trip/period and “not applicable” response. 3. Residents without reliable internet may be omitted; selected invitees may not respond, potentially related to access. 4. Assignment supports causal comparisons among experimental units; sampling supports population representation.
</details>

### Debrief
Write a neutral question; name one coverage and one nonresponse risk.

Record the formula, assumptions, and one follow-up question.